### I'd like to implement an API that shows the current air quality reading for locations in Kentucky using PurpleAir.

In [69]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import requests
import os
from dotenv import load_dotenv
from datetime import datetime
import folium

In [70]:
load_dotenv()

API_KEY = os.getenv("PURPLEAIR_READ_KEY")
BASE_URL = "https://api.purpleair.com/v1"

headers = {
    "X-API-Key": API_KEY
}

# --- Get a single sensor's data ---
sensor_index = 131075  # Replace with your target sensor's index

response = requests.get(
    f"{BASE_URL}/sensors/{sensor_index}",
    headers=headers,
    params={"fields": "name ,pm2.5, humidity, temperature"}
)

if response.status_code == 200:
    data = response.json()
    print(data)
else:
    print(f"Error {response.status_code}:", response.text)

{'api_version': 'V1.2.0-1.1.45', 'time_stamp': 1775425498, 'data_time_stamp': 1775425492, 'sensor': {'sensor_index': 131075, 'name': 'Mariners Bluff', 'humidity': 100, 'temperature': 83, 'pm2.5': 14.9}}


In [71]:
#Now responses for KY
response = requests.get(
    "https://api.purpleair.com/v1/sensors",
    headers=headers,
    params={
        "fields": "name, pm2.5, humidity, temperature, latitude, longitude",
        "location_type": 0,
        "nwlng": -89.57,
        "nwlat": 39.15,
        "selng": -81.96,
        "selat": 36.49,
    }
)

data = response.json()
print(f"Found {len(data['data'])} sensors")
print(data)

Found 119 sensors
{'api_version': 'V1.2.0-1.1.45', 'time_stamp': 1775425498, 'data_time_stamp': 1775425492, 'location_type': 0, 'max_age': 604800, 'firmware_default_version': '7.02', 'fields': ['sensor_index', 'name', 'latitude', 'longitude', 'humidity', 'temperature', 'pm2.5'], 'data': [[262243, 'TDEC APC - Kingsport - 262243', 36.538925, -82.52163, 41, 63, 1.1], [263785, 'EWV Whitman 1', 37.79283, -82.030815, 44, 63, 1.5], [263797, 'EWV Logan 2', 37.904606, -82.10735, 33, 65, 0.6], [263809, 'EWV Point Pleasant 3', 38.87221, -82.12882, 37, 61, 2.0], [273763, 'Boone Block', 39.08691, -84.50917, 36, 59, 1.8], [13031, 'Mount Vernon,Ind.', 37.94211, -87.89055, 24, 79, 1.1], [276356, 'Floyd Street Old Louisville', 38.23082, -85.75195, 26, 68, 2.2], [278523, 'EWV Five Mile 1', 37.742935, -82.14651, 39, 64, 0.7], [278531, 'EWV Logan 3', 37.881184, -82.081055, 38, 61, 0.3], [278566, 'EWV Point Pleasant 1', 38.91047, -82.11349, 44, 57, 1.9], [279406, 'Barbourville, KY / Knox County', 36.91824,

In [72]:
fields = data['fields']
sensors = data['data']

for sensor in sensors:
    sensor_dict = dict(zip(fields, sensor))
    print(f"Name: {sensor_dict['name']}")
    print(f"  PM2.5:       {sensor_dict['pm2.5']}")
    print(f"  Humidity:    {sensor_dict['humidity']}")
    print(f"  Temperature: {sensor_dict['temperature']}")
    print()

Name: TDEC APC - Kingsport - 262243
  PM2.5:       1.1
  Humidity:    41
  Temperature: 63

Name: EWV Whitman 1
  PM2.5:       1.5
  Humidity:    44
  Temperature: 63

Name: EWV Logan 2
  PM2.5:       0.6
  Humidity:    33
  Temperature: 65

Name: EWV Point Pleasant 3
  PM2.5:       2.0
  Humidity:    37
  Temperature: 61

Name: Boone Block
  PM2.5:       1.8
  Humidity:    36
  Temperature: 59

Name: Mount Vernon,Ind.
  PM2.5:       1.1
  Humidity:    24
  Temperature: 79

Name: Floyd Street Old Louisville
  PM2.5:       2.2
  Humidity:    26
  Temperature: 68

Name: EWV Five Mile 1
  PM2.5:       0.7
  Humidity:    39
  Temperature: 64

Name: EWV Logan 3
  PM2.5:       0.3
  Humidity:    38
  Temperature: 61

Name: EWV Point Pleasant 1
  PM2.5:       1.9
  Humidity:    44
  Temperature: 57

Name: Barbourville, KY / Knox County
  PM2.5:       0.4
  Humidity:    31
  Temperature: 67

Name: Cincinnati State
  PM2.5:       1.9
  Humidity:    37
  Temperature: 58

Name: Cincy Air Watch- Z

### The API now gives all sensor readings available in Kentucky, about 119 so far until more people buy/register their PurpleAir sensors. With that, we can now make a map using Folium to show air quality readings across KY.

In [73]:
#Center map on Kentucky
m = folium.Map(location=[37.8, -85.8], zoom_start=7)

for sensor in sensors:
    sensor_dict = dict(zip(fields, sensor))
    
    folium.Marker(
        location=[sensor_dict['latitude'], sensor_dict['longitude']],
        popup=folium.Popup(
            f"<b>{sensor_dict['name']}</b><br>"
            f"PM2.5: {sensor_dict['pm2.5']}<br>"
            f"Humidity: {sensor_dict['humidity']}<br>"
            f"Temperature: {sensor_dict['temperature']}",
            max_width=200
        ),
        tooltip=sensor_dict['name']
    ).add_to(m)

m.save("../End_Visualizations/kentucky_sensors_map.html")
print("Map saved.")

Map saved.


### Now after having made the map, we can make a unique function that finds sensors based on the area's air quality. We put in the numerical threshold we're looking for and the given output will be sensors correlating with what we're searching for.

In [74]:
def get_sensors_by_air_quality(threshold):
    """
    Returns sensors where PM2.5 exceeds the given threshold.
    
    Parameters:
        threshold (float): PM2.5 value to filter by
    
    Returns:
        list of dicts with sensor data
    """
    response = requests.get(
        "https://api.purpleair.com/v1/sensors",
        headers=headers,
        params={
            "fields": "name,pm2.5,humidity,temperature,latitude,longitude",
            "location_type": 0,
            "nwlng": -89.57,
            "nwlat": 39.15,
            "selng": -81.96,
            "selat": 36.49,
        }
    )

    data = response.json()
    fields = data['fields']
    sensors = data['data']

    results = []
    for sensor in sensors:
        sensor_dict = dict(zip(fields, sensor))
        if sensor_dict['pm2.5'] is not None and sensor_dict['pm2.5'] > threshold:
            results.append(sensor_dict)

    print(f"Found {len(results)} sensors above PM2.5 threshold of {threshold}")
    return results

# Example usage — EPA's "Good" air quality limit is 12.0
flagged_sensors = get_sensors_by_air_quality(12.0)
for sensor in flagged_sensors:
    print(f"Name: {sensor['name']}, PM2.5: {sensor['pm2.5']}")

Found 1 sensors above PM2.5 threshold of 12.0
Name: AV60-4E20, PM2.5: 16.3


##### This API was made with the help of Claude AI in some parts to create a workable map and clarify how the API from PurpleAir was meant to be set up and used as it differs from a basic setup I did in my API weather project that I used for comparison.